# 11 – Recommendations

Esplorazione e data cleaning del dataset `recommendations.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `mal_id` | ID dell'anime su MAL |
| `recommendation_mal_id` | ID dell'anime raccomandato su MAL |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [27]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rec = pd.read_csv('../datasets/recommendations.csv')
print(f'Shape: {df_rec.shape}')
df_rec.info()
df_rec.head()

Shape: (105249, 2)
<class 'pandas.DataFrame'>
RangeIndex: 105249 entries, 0 to 105248
Data columns (total 2 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   mal_id                 105249 non-null  int64
 1   recommendation_mal_id  105249 non-null  int64
dtypes: int64(2)
memory usage: 1.6 MB


,mal_id,recommendation_mal_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681


**Osservazioni iniziali:**
- Il dataset contiene **105.249 righe** e **2 colonne**.
- Tutte e 2 le colonne sono complete: **nessun valore nullo**.
- I tipi di dati sono adeguati: `int64` per entrambi gli ID.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [28]:
n_originale = len(df_rec)

mask_dup = df_rec.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_rec[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_rec.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_rec):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 105,249
Righe dopo la rimozione     : 105,249


Nessun duplicato esatto trovato. Tutte le 105.249 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: lo stesso anime può comparire più volte come sorgente di più raccomandazioni diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [29]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_mal = check_fk(df_rec['mal_id'], df_details['mal_id'], child_df=df_rec)

print(f'Null in mal_id               : {df_rec["mal_id"].isna().sum()}')
print(f'Duplicati in mal_id (attesi) : {df_rec["mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       105,249
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            105,249  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              105,249  (100.00%)
  ✗  Righe orfane (FK non in PK)      0  (0.00%)
     → ID orfani unici                0

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

Nessuna violazione. Tutti i valori FK esistono nella tabella padre.

Null in mal_id               : 0
Duplicati in mal_id (attesi) : 96,216


**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime sorgente valido.
- **Integrità referenziale**: non ci sono righe orfane.

Nessuna pulizia necessaria. 

### 2.2 `recommendation_mal_id`

Questa colonna è anch'essa una **chiave esterna** verso la stessa tabella padre `details.csv`, ma referenzia l'anime raccomandato.

I valori duplicati sono **attesi**: lo stesso anime può essere raccomandato a partire da diversi anime.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Riutilizziamo `df_details` già caricato nella sezione precedente.

In [30]:
mask_orphan_rec = check_fk(df_rec['recommendation_mal_id'], df_details['mal_id'], child_df=df_rec)

print(f'Null in recommendation_mal_id               : {df_rec["recommendation_mal_id"].isna().sum()}')
print(f'Duplicati in recommendation_mal_id (attesi) : {df_rec["recommendation_mal_id"].duplicated().sum():,}')


  Colonna FK  (tabella figlia)        recommendation_mal_id
  Colonna PK  (tabella padre)         mal_id
────────────────────────────────────────────────────────────────────────────────

Riepilogo conteggi
────────────────────────────────────────────────────────────────────────────────

  Righe totali (tabella figlia)       105,249
  Valori null nella FK                0  (0.00%)
  Valori non-null nella FK            105,249  (100.00%)
  Valori unici nella PK padre         28,955

  ✓  Righe con FK valida              105,122  (99.88%)
  ✗  Righe orfane (FK non in PK)      127  (0.12%)
     → ID orfani unici                12

Valutazione integrità referenziale
────────────────────────────────────────────────────────────────────────────────

127 riga/e (0.12%) violano l'integrità referenziale.

Campione righe orfane (prime 10)
────────────────────────────────────────────────────────────────────────────────

      mal_id  recommendation_mal_id
1336   11759                   6408
1613  

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime raccomandato valido.
- **Integrità referenziale**: ci sono 127 righe orfane che vengono rimosse nella cella seguente.

In [31]:
if mask_orphan_rec.any():
    n_orfane = mask_orphan_rec.sum()
    df_rec = df_rec[~mask_orphan_rec].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane}')
    print(f'Righe rimanenti      : {len(df_rec):,}')
else:
    print('Nessuna riga orfana da rimuovere.')

Righe orfane rimosse : 127
Righe rimanenti      : 105,122


### 2.3 Chiave primaria `(mal_id, recommendation_mal_id)`

La coppia `(mal_id, recommendation_mal_id)` è la chiave primaria naturale: lo stesso paio di anime non dovrebbe comparire più di una volta. Verifichiamo che sia univoca dopo la pulizia.

In [32]:
n_pk_dup = df_rec.duplicated(subset=['mal_id', 'recommendation_mal_id'], keep=False).sum()
print(f'Duplicati su (mal_id, recommendation_mal_id): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_rec.drop_duplicates(subset=['mal_id', 'recommendation_mal_id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_rec):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (mal_id, recommendation_mal_id): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


La chiave primaria è univoca dopo la pulizia. Il dataset è pronto per il salvataggio.

## 3. Riepilogo e Salvataggio
Le operazioni di pulizia sono state effettuate colonna per colonna nella sezione 2. In questa sezione riepiloghiamo il risultato ed effettuiamo il salvataggio del dataset finale.

In [33]:
print('Riepilogo Dataset Pulito')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_rec):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_rec):>10,}')
print()
df_rec.to_csv('../datasets_cleaned/recommendations_clean.csv', index=False)
print('Salvato: datasets_cleaned/recommendations_clean.csv')

Riepilogo Dataset Pulito
Righe originali      :    105,249
Righe dopo cleaning  :    105,122
Righe rimosse totali :        127



Salvato: datasets_cleaned/recommendations_clean.csv
